<a href="https://colab.research.google.com/github/AmaimaKhalidSethi/LLM_ENGINEERING/blob/main/Week3/task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gradio pandas pydantic groq google-generativeai transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00


In [3]:
import os
import json
import time
import torch
import pandas as pd
import gradio as gr
from google.colab import userdata
from pydantic import BaseModel, Field
from typing import List, Literal
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
import google.generativeai as genai
from groq import Groq

# --- CONFIGURATION ---
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
# Using Qwen 1.5B: It's fast, smart, and doesn't require a gated HF token.
HF_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# --- DATA SCHEMA ---
class SIEMLog(BaseModel):
    timestamp: str = Field(..., description="ISO 8601 timestamp")
    source_ip: str = Field(..., description="IPv4 address")
    http_method: Literal["GET", "POST", "PUT", "DELETE"]
    endpoint: str = Field(..., description="Accessed URL path")
    status_code: int = Field(..., description="HTTP Status (200, 404, etc.)")
    is_malicious: bool = Field(..., description="True if this is an attack")
    attack_type: str = Field(..., description="Type of attack if malicious, else 'None'")

# --- THE GENERATOR ENGINE ---
class LogGenerator:
    def __init__(self):
        # Cloud Clients
        self.groq_client = Groq(api_key=GROQ_API_KEY)
        genai.configure(api_key=GEMINI_API_KEY)
        self.gemini_model = genai.GenerativeModel("gemini-1.5-flash")

        # Local HF Model Setup (Quantized)
        print(f"Loading {HF_MODEL_ID} in 4-bit...")
        self.bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )
        self.tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
        self.hf_model = AutoModelForCausalLM.from_pretrained(
            HF_MODEL_ID,
            quantization_config=self.bnb_config,
            device_map="auto"
        )
        self.hf_pipe = pipeline("text-generation", model=self.hf_model, tokenizer=self.tokenizer)

    def get_prompt(self, count, description):
        schema_json = json.dumps(SIEMLog.model_json_schema(), indent=2)
        return f"""Generate exactly {count} synthetic SIEM logs for this scenario: {description}
        Output must be a valid JSON array of objects strictly following this schema:
        {schema_json}
        Return ONLY the JSON array."""

    def generate(self, provider, count, description):
        prompt = self.get_prompt(count, description)
        start_time = time.time()

        try:
            if "Groq" in provider:
                resp = self.groq_client.chat.completions.create(
                    model="llama-3.1-8b-instant",
                    messages=[{"role": "user", "content": prompt}],
                    response_format={"type": "json_object"}
                )
                raw_data = resp.choices[0].message.content

            elif "Gemini" in provider:
                resp = self.gemini_model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})
                raw_data = resp.text

            elif "HuggingFace" in provider:
                messages = [{"role": "user", "content": prompt}]
                # Note: We use a short max_new_tokens to keep it fast for testing
                outputs = self.hf_pipe(messages, max_new_tokens=1024, do_sample=True, temperature=0.7)
                raw_data = outputs[0]["generated_text"][-1]["content"]

            logs = json.loads(raw_data)
            if isinstance(logs, dict): logs = list(logs.values())[0] # Cleanup logic
            return logs, round(time.time() - start_time, 2)

        except Exception as e:
            return [{"error": str(e)}], 0

In [4]:
class LogGenerator:
    def __init__(self):
        self.groq_client = Groq(api_key=GROQ_API_KEY)
        genai.configure(api_key=GEMINI_API_KEY)
        self.gemini_model = genai.GenerativeModel(MODELS["Gemini (Cloud - Smart)"])

        # Local HF Model Setup (Quantized)
        print(f"Loading {HF_MODEL_ID} in 4-bit...")
        self.bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )
        self.tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
        self.hf_model = AutoModelForCausalLM.from_pretrained(
            HF_MODEL_ID,
            quantization_config=self.bnb_config,
            device_map="auto"
        )
        self.hf_pipe = pipeline("text-generation", model=self.hf_model, tokenizer=self.tokenizer)

    def get_prompt(self, count, description):
        schema_json = json.dumps(SIEMLog.model_json_schema(), indent=2)
        return f"""
        Generate exactly {count} synthetic SIEM logs based on this scenario: {description}
        Output MUST be a valid JSON array of objects strictly following this JSON Schema:
        {schema_json}
        Return ONLY the JSON array. No preamble or conversational text.
        """

    def generate(self, provider, count, description):
        prompt = self.get_prompt(count, description)
        start_time = time.time()

        try:
            if "Groq" in provider:
                response = self.groq_client.chat.completions.create(
                    model=MODELS["Groq (Cloud - Fast)"],
                    messages=[{"role": "user", "content": prompt}],
                    response_format={"type": "json_object"}
                )
                raw_data = response.choices[0].message.content

            elif "Gemini" in provider:
                response = self.gemini_model.generate_content(
                    prompt,
                    generation_config={"response_mime_type": "application/json"}
                )
                raw_data = response.text

            elif "HuggingFace" in provider:
                messages = [{"role": "user", "content": prompt}]
                outputs = self.hf_pipe(messages, max_new_tokens=1500, do_sample=True, temperature=0.7)
                raw_data = outputs[0]["generated_text"][-1]["content"]

            # Process and Clean
            logs = json.loads(raw_data)
            if isinstance(logs, dict):
                logs = list(logs.values())[0]

            latency = round(time.time() - start_time, 2)
            return logs, latency

        except Exception as e:
            return [{"error": str(e)}], 0

In [5]:
MODELS = {
    "Groq (Cloud - Fast)": "llama-3.1-8b-instant",
    "Gemini (Cloud - Smart)": "gemini-1.5-flash",
    "HuggingFace (Local - Qwen)": HF_MODEL_ID
}

generator = LogGenerator()

def master_interface(provider, count, scenario):
    all_logs = []
    total_latency = 0

    batch_size = 5 # Reduced batch size for local inference stability
    loops = (count // batch_size) + (1 if count % batch_size > 0 else 0)

    for _ in range(loops):
        current_count = min(batch_size, count - len(all_logs))
        logs, lat = generator.generate(provider, current_count, scenario)
        all_logs.extend(logs)
        total_latency += lat

    df = pd.DataFrame(all_logs)
    stats = f"Model: {provider} | Total Time: {total_latency}s | Records: {len(df)}"
    df.to_csv("synthetic_logs.csv", index=False)
    return df, stats, "synthetic_logs.csv"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛡️ CyberSynth: Synthetic SIEM Generator")
    gr.Markdown("Build realistic security datasets using Cloud APIs or Local HuggingFace Models.")

    with gr.Row():
        with gr.Column(scale=1):
            provider = gr.Radio(list(MODELS.keys()), label="Select Model Provider", value="Groq (Cloud - Fast)")
            scenario = gr.Textbox(label="Dataset Scenario", placeholder="SQL injection attempts on login endpoint...", lines=3)
            count = gr.Slider(5, 50, step=5, label="Number of Records")
            btn = gr.Button("🚀 Generate & Export", variant="primary")

        with gr.Column(scale=2):
            stats_output = gr.Label(label="Performance Metrics")
            table_output = gr.Dataframe()
            file_output = gr.File(label="Download CSV")

    btn.click(master_interface, inputs=[provider, count, scenario], outputs=[table_output, stats_output, file_output])

demo.launch()

Loading Qwen/Qwen2.5-1.5B-Instruct in 4-bit...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/tmp/ipykernel_13409/578301363.py:27: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d25270685dbf919a87.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
